# K-Means Clustering 

## 1. What is K-Means?
K-Means is an iterative, centroid-based partitioning algorithm used for **Unsupervised Learning**. 
* **Unsupervised:** No labels are given; the algorithm finds the underlying structure grouping similar data points by itself.
* **Centroid-based:** Each cluster is represented by a center point (the "mean" or average of the points).
* **Partitioning:** It divides $n$ observations into $k$ distinct clusters.
* **Distance minimization:** Points are assigned to the nearest centroid.
* **Iterative refinement:** The process repeats (assign points $\rightarrow$ move centroid $\rightarrow$ assign points) until the centroids stop moving (convergence).



**Core Assumptions:**
K-Means assumes that clusters are spherical, equally sized, and have similar densities.

---

## 2. The Objective Function & Complexity
The goal of K-Means is to minimize the within-cluster variance, also known as **Inertia** or **Within-Cluster Sum of Squares (WCSS)**.

$$Inertia = \sum_{j=1}^{k} \sum_{i=1}^{n} ||x_i - \mu_j||^2$$

* $x_i$ = a data point in cluster $j$
* $\mu_j$ = the centroid of cluster $j$

### Computational Complexity
* **Time Complexity:** $\mathcal{O}(n \times k \times d \times i)$
* **Space Complexity:** $\mathcal{O}(n \times k + k \times d)$
*(where $n$ = data points, $k$ = clusters, $d$ = dimensions/features, $i$ = iterations)*

---

## 3. Initialization: Random vs. K-Means++
Choosing the initial starting points for the centroids is critical.

* **Random Initialization:** Can lead to bad clusters or slow convergence if centroids start too close to each other.
* **K-Means++ (Smart Initialization):** Picks the first centroid randomly, but chooses subsequent centroids to be as *far away* from existing centroids as possible. This leads to much faster convergence and better cluster formations.



### Python Implementation Parameters (Scikit-Learn)
```python
from sklearn.cluster import KMeans

model = KMeans(
    n_clusters=8,           # Most important: Choose K manually
    init='k-means++',       # Smart initialization (default)
    n_init=10,              # Run 10 times with different inits, keep the best
    max_iter=300,           # Maximum iterations per run
    tol=1e-4,               # Convergence tolerance (when to stop)
    random_state=42,        # Seed for reproducibility
    algorithm='auto'        # 'elkan' is faster for dense data
)
```

## 4. When K-Means FAILS (Very Important)

Because K-Means relies heavily on the geometric distance from a center point, it struggles under specific data conditions:

- Non-spherical clusters:  Moon-shaped or elongated data.

- Different cluster sizes: One massive cluster next to a tiny one.

- Different densities: Tightly packed clusters next to sparse ones.

- Outliers: Centroids are highly sensitive to extreme values (they get pulled toward the outlier).

---

## 5. Best Practices & Evaluation

**Feature Scaling is Mandatory**

Because K-Means calculates geometric distances (usually Euclidean), features with larger ranges will dominate the calculation. You must scale your features ( using StandardScaler or MinMaxScaler) before running the algorithm.

**Evaluation Metrics**

Since there are no true labels to check against (unsupervised), we evaluate the cluster quality using:
1. Inertia (WCSS): Used in the "Elbow Method" to find the optimal $K$.

2. Silhouette Score: Measures how similar an object is to its own cluster compared to other clusters (ranges from -1 to 1).
3. Davies-Bouldin Index: Lower values indicate better clustering (clusters are distinct and well-separated).

## 6. K-Means Algorithm Steps (Crucial for Interviews)
Understanding the exact sequence of operations is fundamental for exams and technical interviews.

1. **Choose K:** Decide the target number of clusters.
2. **Initialize:** Place $K$ centroids randomly in the data space (or use K-Means++).
3. **Assign:** Calculate the distance from every data point to all centroids, and assign each point to its *nearest* centroid.
4. **Update:** Recalculate the position of each centroid by computing the mean (average) of all the data points currently assigned to it.
5. **Repeat:** Repeat the *Assign* and *Update* steps iteratively.
6. **Stop:** The algorithm terminates when either:
    * The centroids stop moving (convergence is reached).
    * The maximum number of iterations (`max_iter`) is reached.



---

## 7. How to Choose K: The Elbow Method
Since K-Means requires you to define $K$ beforehand, the **Elbow Method** is the most common visual tool used to find the optimal number of clusters.

**The Steps:**
1. Train multiple K-Means models for different values of $K$ (e.g., $K = 1, 2, 3 \dots 10$).
2. Compute the **WCSS** (Within-Cluster Sum of Squares, or Inertia) for each model.
3. Plot $K$ (x-axis) vs. WCSS (y-axis).
4. Identify the "elbow" point—the point where the curve bends sharply.

**The Reasoning:** Before the elbow point, adding more clusters dramatically reduces the variance (WCSS). After the elbow point, the drop in variance becomes marginal, meaning you are just artificially splitting valid clusters into smaller, unnecessary pieces.



---

## 8. The Core Distance Metric
Because K-Means partitions data strictly based on physical proximity in vector space, the choice of distance metric is the backbone of the algorithm. Almost all standard implementations use **Euclidean Distance**:

$$d(x,y) = \sqrt{\sum (x_i - y_i)^2}$$

*(Note: Because this formula squares the differences in magnitudes, features with larger scales will completely dominate the calculation. This is exactly why feature scaling is absolutely mandatory before running K-Means).*

---

## 9. Hard Clustering vs. Soft Clustering
K-Means is strictly a **Hard Clustering** algorithm.

* **Hard Clustering:** A binary relationship. Each point belongs to *exactly one* cluster. There is no overlap, doubt, or probability.
* **Soft Clustering:** A probabilistic relationship. A point can belong to multiple clusters with varying degrees of certainty (e.g., 80% Cluster A, 20% Cluster B).

| Algorithm | Type | Description |
| :--- | :--- | :--- |
| **K-Means** | Hard Clustering | Absolute assignment based purely on the nearest centroid. |
| **Gaussian Mixture Models (GMM)** | Soft Clustering | Probabilistic assignment based on underlying data distributions. |

---

## 10. Optimization: Mini-Batch K-Means
Standard K-Means uses the *entire dataset* at every single iteration to recalculate centroids. If you have millions of rows, this becomes incredibly slow. 

**Mini-Batch K-Means** is a variation designed for massive datasets:
* **How it works:** Instead of the full dataset, it uses small, randomly sampled batches of data during each iteration to update the centroids.
* **Benefits:**
    * Drastically **faster** training times.
    * Much **lower memory usage** (you do not need to load the entire dataset into RAM at once).
    * The resulting clusters are usually almost identical in quality to standard K-Means.

In [6]:
import random
import math
def euclidean_distance(p1, p2):
    return math.sqrt(sum((a - b) ** 2 for a, b in zip(p1, p2)))
def assign_clusters(X, centroids):
    clusters = [[] for _ in centroids]
    
    for point in X:
        distances = [euclidean_distance(point, c) for c in centroids]
        cluster_index = distances.index(min(distances))
        clusters[cluster_index].append(point)
    
    return clusters
def update_centroids(clusters):
    new_centroids = []
    
    for cluster in clusters:
        centroid = [
            sum(dim) / len(cluster)
            for dim in zip(*cluster)
        ]
        new_centroids.append(centroid)
    
    return new_centroids
def kmeans(X, k, max_iters=100):
    centroids = random.sample(X, k)

    for _ in range(max_iters):
        clusters = assign_clusters(X, centroids)
        new_centroids = update_centroids(clusters)

        if new_centroids == centroids:
            break

        centroids = new_centroids

    return clusters, centroids
X = [
    [1, 2], [1, 4], [1, 0],
    [10, 2], [10, 4], [10, 0]
]

clusters, centroids = kmeans(X, k=2)

print("Centroids:", centroids)
print("Clusters:", clusters)


Centroids: [[1, 2], [10, 2]]
Clusters: [[[1, 2], [1, 4], [1, 0]], [[10, 2], [10, 4], [10, 0]]]


In [7]:
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
X = np.array([
    [1, 2], [1, 4], [1, 0],
    [10, 2], [10, 4], [10, 0]
])
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
kmeans = KMeans(
    n_clusters=2,
    init='k-means++',
    random_state=42
)

kmeans.fit(X_scaled)
labels = kmeans.labels_
centroids = kmeans.cluster_centers_

print("Labels:", labels)
print("Centroids:", centroids)


Labels: [0 1 0 1 1 0]
Centroids: [[-0.33333333 -0.81649658]
 [ 0.33333333  0.81649658]]


In [9]:
from sklearn.metrics import silhouette_score

score = silhouette_score(X_scaled, labels)
print("Silhouette Score:", score)


Silhouette Score: 0.15910418698883563


WHY use K-Means?
To find structure in unlabeled data

When:

You don’t have labels

You still want to group similar data

Example
You have customers, but no “type” column
→ K-Means discovers customer segments automatically

Simple, fast, and scalable 

Why companies love it:

Easy to understand

Very fast even for large datasets

Works well in high dimensions (with scaling)

Used in:

Industry

Interviews

Baseline clustering

Clear objective function

K-Means minimizes within-cluster distance (WCSS)

This makes it:

Mathematically clean

Easy to optimize

Easy to explain

Easy to interpret results

Output is:

Cluster labels

Centroids (means)

Centroid = “average representative” of group

Good baseline model

Before trying complex clustering:

Try K-Means first

If it works → great

If not → move to DBSCAN / Hierarchical

WHEN to use K-Means?

Use K-Means ONLY if these conditions are mostly true 

Data is numerical

K-Means uses distance.

Height, weight, income
Color, category, text (without encoding)

Clusters are roughly spherical

K-Means assumes:

Round-shaped clusters

Equal spread

Works well when clusters look like “balls”

Similar cluster sizes

If one cluster is huge and one is tiny →  bad

ou know (or can estimate) K

You can:

Use Elbow Method

Use Silhouette Score

Use domain knowledge

Few outliers

Outliers pull centroids badly

Always check:

Boxplot

Scatter plot

WHEN NOT to use K-Means 
Non-spherical clusters

Example:

Moon shape

Spiral shape

Use DBSCAN

Different densities

One cluster dense, another sparse

K-Means fails

Categorical data only

Distance doesn’t make sense

Use:

K-Modes

Hierarchical clustering

Many outliers

Centroid shifts incorrectly
Clean data or use DBSCAN


Customer Segmentation
Features: age, income, spending_score
Why K-Means?
✔ Numeric
✔ Want segments
✔ Fast

Image Compression
Sales, visits, revenue

Document Clustering (after TF-IDF)
Vectors are numeric → K-Means works

Why do you use K-Means?
K-Means is used to cluster unlabeled numerical data by minimizing within-cluster variance. It is fast, simple, scalable, and works well when clusters are spherical and of similar size.

When do you use K-Means?
I use K-Means when the data is numerical, scaled, has few outliers, and when the number of clusters can be estimated using methods like the elbow or silhouette score.

K-Means = fast + numeric + spherical + known K



